# Committee Session Subject Labeling
Segments each committee session JSON by topic using DICTA-BERT embeddings for boundary detection
and dicta-il/dicta-lm (2B, quantized) for generating Hebrew subject labels.

Output mirrors `ksafim_subject_labeling.csv` but uses utterance ID ranges instead of page numbers.

In [ ]:
!pip install -q transformers accelerate bitsandbytes sentencepiece torch tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import json
import glob

# --- CONFIGURE THESE ---
# Path to the committee_data folder on your Drive (or local if running locally)
COMMITTEE_DATA_PATH = "/content/drive/MyDrive/KNESSET_DATA/committee_data"

# Output CSV path
OUTPUT_CSV = "/content/drive/MyDrive/KNESSET_DATA/committee_subject_labeling.csv"

# Segmentation sensitivity: higher = fewer, coarser segments; lower = more fine-grained
SIMILARITY_THRESHOLD = 0.75  # cosine similarity drop below this triggers a new segment
MIN_UTTERANCES_PER_SEGMENT = 3  # ignore segments shorter than this
WINDOW_SIZE = 4  # utterances per side for smoothing

# DICTA models
EMBED_MODEL_NAME = "dicta-il/dictabert"        # for boundary detection
GEN_MODEL_NAME   = "dicta-il/dicta-lm"         # 2B generative model for labeling
USE_4BIT_QUANT   = True                        # quantize gen model to 4-bit
# -----------------------

json_files = sorted(glob.glob(os.path.join(COMMITTEE_DATA_PATH, "**", "*.json"), recursive=True))
print(f"Found {len(json_files)} session files")

In [ ]:
# Inspect a sample file
with open(json_files[0], encoding="utf-8") as f:
    sample = json.load(f)

print("doc_id:", sample["doc_id"])
print("committee:", sample["committee"])
print("date:", sample["date"])
print("utterances count:", len(sample["utterances"]))
print("first utterance:", sample["utterances"][0])

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel
import numpy as np

embed_tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL_NAME)
embed_model = AutoModel.from_pretrained(EMBED_MODEL_NAME)
embed_model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
embed_model = embed_model.to(device)
print(f"Embedding model loaded on {device}")

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16) if USE_4BIT_QUANT else None

gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)
gen_model = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto",
)
gen_model.eval()
print("Generative model loaded")

In [ ]:
def mean_pool(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return (token_embeddings * mask).sum(1) / mask.sum(1).clamp(min=1e-9)


@torch.no_grad()
def embed_texts(texts: list, batch_size: int = 16) -> np.ndarray:
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        encoded = embed_tokenizer(
            batch, padding=True, truncation=True, max_length=512, return_tensors="pt"
        ).to(device)
        output = embed_model(**encoded)
        embeddings = mean_pool(output, encoded["attention_mask"])
        all_embeddings.append(embeddings.cpu().numpy())
    return np.vstack(all_embeddings)


def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))


def detect_boundaries(embeddings: np.ndarray, window: int, threshold: float) -> list:
    """
    TextTiling-style boundary detection.
    Returns list of utterance indices where a new segment starts (always includes 0).
    """
    n = len(embeddings)
    if n <= window * 2:
        return [0]

    scores = []
    for i in range(window, n - window):
        left  = embeddings[max(0, i - window):i].mean(axis=0)
        right = embeddings[i:i + window].mean(axis=0)
        scores.append(cosine_sim(left, right))

    arr = np.array(scores)
    if arr.std() > 0:
        arr = (arr - arr.mean()) / arr.std()

    boundaries = [0]
    for idx, score in enumerate(arr):
        utterance_idx = idx + window
        if score < -threshold:
            boundaries.append(utterance_idx)
    return boundaries


def segment_session(utterances: list) -> list:
    """
    Returns list of (start_id, end_id) tuples using the 'id' field from each utterance.
    """
    if not utterances:
        return []

    texts = [u["speaker"] + ": " + u["text"] for u in utterances]
    ids   = [int(u["id"]) for u in utterances]

    if len(utterances) < MIN_UTTERANCES_PER_SEGMENT * 2:
        return [(ids[0], ids[-1])]

    embeddings = embed_texts(texts)
    raw_boundaries = detect_boundaries(embeddings, WINDOW_SIZE, SIMILARITY_THRESHOLD)

    merged = [raw_boundaries[0]]
    for b in raw_boundaries[1:]:
        if b - merged[-1] >= MIN_UTTERANCES_PER_SEGMENT:
            merged.append(b)

    segments = []
    for i, start in enumerate(merged):
        end = merged[i + 1] - 1 if i + 1 < len(merged) else len(utterances) - 1
        segments.append((ids[start], ids[end]))
    return segments


print("Segmentation helpers defined")

In [ ]:
def build_label_prompt(utterances: list, doc_title: str) -> str:
    excerpt_lines = []
    for u in utterances[:12]:
        line = f"{u['speaker']}: {u['text'][:200]}"
        excerpt_lines.append(line)
    excerpt = "\n".join(excerpt_lines)

    return (
        f"הנושא הכללי של הישיבה: {doc_title}\n\n"
        "להלן קטע מפרוטוקול ועדה בכנסת ישראל:\n"
        f"{excerpt}\n\n"
        "תאר בביטוי עברי קצר (עד 6 מילים) את הנושא הספציפי שנידון בקטע זה.\n"
        "נושא:"
    )


@torch.no_grad()
def generate_label(prompt: str, max_new_tokens: int = 20) -> str:
    inputs = gen_tokenizer(prompt, return_tensors="pt").to(gen_model.device)
    output_ids = gen_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=1.0,
        pad_token_id=gen_tokenizer.eos_token_id,
    )
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    label = gen_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return label.split("\n")[0].split(".")[0].strip()


print("Label generation helpers defined")

In [ ]:
import pandas as pd
from tqdm.auto import tqdm

rows = []

for json_path in tqdm(json_files, desc="Sessions"):
    with open(json_path, encoding="utf-8") as f:
        session = json.load(f)

    doc_id     = session.get("doc_id", "")
    committee  = session.get("committee", "")
    date       = session.get("date", "")
    title      = session.get("title", "")
    utterances = session.get("utterances", [])

    if not utterances:
        continue

    segments = segment_session(utterances)
    id_to_utt = {int(u["id"]): u for u in utterances}

    for seg_start, seg_end in tqdm(segments, desc=doc_id, leave=False):
        seg_utts = [id_to_utt[i] for i in range(seg_start, seg_end + 1) if i in id_to_utt]
        if not seg_utts:
            continue

        prompt = build_label_prompt(seg_utts, title)
        label  = generate_label(prompt)

        rows.append({
            "doc_id":         doc_id,
            "committee_name": committee,
            "session_date":   date,
            "utterance_from": seg_start,
            "utterance_to":   seg_end,
            "subject_label":  label,
            "session_title":  title,
        })

print(f"\nTotal segments labelled: {len(rows)}")

In [ ]:
df = pd.DataFrame(rows, columns=[
    "doc_id", "committee_name", "session_date",
    "utterance_from", "utterance_to", "subject_label", "session_title"
])

df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")  # utf-8-sig for Excel Hebrew compat
print(f"Saved {len(df)} rows to {OUTPUT_CSV}")
df.head(10)

In [ ]:
# Sanity-check: show all segments for a single session
SAMPLE_DOC_ID = df["doc_id"].iloc[0]
subset = df[df["doc_id"] == SAMPLE_DOC_ID]

print(f"Session: {SAMPLE_DOC_ID}")
print(f"Segments: {len(subset)}\n")
for _, row in subset.iterrows():
    print(f"  utterances {row.utterance_from:>4}–{row.utterance_to:<4}  →  {row.subject_label}")